# 05 Mask-Cluster / SAM-LAD Relation Probe

역할: segmentation raw mask를 relation graph component node로 정규화하는 probe를 관리한다. 기본 경로는 이미 생성된 `grounding_mask.png`를 빠르게 clustering하는 `grounding_mask_cluster`이고, 필요하면 `sam_lad`로 바꿔 SAM-HQ automatic mask proposal을 새로 생성한다.


## Cell Role: Probe Mode And Runtime Guard

`COMPONENT_MODEL`을 먼저 정한다. `grounding_mask_cluster`는 torch/SAM을 import하지 않으므로 NumPy guard를 건너뛴다. `sam_lad`는 torch를 import하므로 Colab NumPy 2.x 런타임에서는 NumPy 1.26.4 설치 후 런타임을 재시작한다.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

COMPONENT_MODEL = 'grounding_mask_cluster'  # 'grounding_mask_cluster' or 'sam_lad'
TARGET_NUMPY = '1.26.4'

if COMPONENT_MODEL == 'sam_lad':
    try:
        numpy_version = importlib.metadata.version('numpy')
    except importlib.metadata.PackageNotFoundError:
        numpy_version = '-'

    print('numpy =', numpy_version)
    if numpy_version == '-' or numpy_version.split('.', maxsplit=1)[0] != '1':
        subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--upgrade',
                '--force-reinstall',
                '--no-cache-dir',
                f'numpy=={TARGET_NUMPY}',
            ],
            check=True,
        )
        raise SystemExit(
            'Installed numpy==1.26.4. Restart the Colab runtime once, then rerun this notebook from the top.'
        )
    print('numpy runtime is compatible with this torch/SAM stack')
else:
    print(f'component_model={COMPONENT_MODEL}; torch/SAM numpy guard skipped')


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM`을 최신 `main` 기준으로 맞춘다. 로컬 notebook 변경과 충돌하지 않도록 relation source, clustering config, 관련 tests만 targeted sync한다.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'

SYNC_PATHS = [
    'experiments/validation/condition_shift_baseline/src/relation',
    'experiments/validation/condition_shift_baseline/configs/grounding_mask_cluster.yaml',
    'experiments/validation/condition_shift_baseline/tests/test_grounding_mask_cluster.py',
]

def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )

colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        local_rev = subprocess.check_output(
            ['git', '-C', str(colab_checkout), 'rev-parse', '--short', 'HEAD'],
            text=True,
        ).strip()
        fetched_rev = subprocess.check_output(
            ['git', '-C', str(colab_checkout), 'rev-parse', '--short', 'FETCH_HEAD'],
            text=True,
        ).strip()
        print(f'local HEAD={local_rev} fetched main={fetched_rev}')
        if local_rev != fetched_rev:
            print('sync relation/config/test files from fetched main without touching local notebooks')
            subprocess.run(
                ['git', '-C', str(colab_checkout), 'checkout', 'FETCH_HEAD', '--', *SYNC_PATHS],
                check=True,
            )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and is_regram_root(p)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for source_root in (SRC_ROOT, SRC_ROOT / 'relation'):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Input Readiness

선택한 component model에 필요한 입력을 확인한다. Colab session에 raw LOCO가 없으면 Drive tar에서 복구한다. `grounding_mask_cluster`는 실행 대상 manifest row에 필요한 `grounding_mask.png`만 Drive에서 session으로 선택 복사한다.


In [ ]:
CATEGORY = 'breakfast_box'
SEVERITY = 'high'
LIMIT = 30
MAX_COMPONENTS = 12
PROGRESS_EVERY = 1

# SAM-LAD-only settings
SAM_DEVICE = 'auto'
SAM_MIN_AREA_RATIO = 0.0005
SAM_SMALL_AREA_RATIO = 0.006
SAM_MAX_AREA_RATIO = 0.85
SAM_POINTS_PER_SIDE = 16  # 16 for smoke/quick run; 32 for fuller SAM-LAD masks
SAM_CROP_N_LAYERS = 0  # 0 is much faster; 1 is closer to SAM-LAD-style dense proposals

POSITION_MANIFEST = REPO_ROOT / 'manifests' / 'query_position_shift.jsonl'
RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
RAW_CATEGORY_GOOD = RAW_LOCO_ROOT / CATEGORY / 'test' / 'good'

GROUNDING_MASK_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'masks' / 'mvtec_loco_caption'
GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT
GROUNDING_CLUSTER_CONFIG = EXP_ROOT / 'configs' / 'grounding_mask_cluster.yaml'
GROUNDING_MASK_CATEGORY_ROOT = GROUNDING_MASK_ROOT / CATEGORY / 'test' / 'good'

SAM_CHECKPOINT = REPO_ROOT / 'external' / 'UniVAD' / 'pretrained_ckpts' / 'sam_hq_vit_h.pth'
SAM_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'models'

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')
DRIVE_MASK_ROOT = Path('/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption')
DRIVE_MODEL_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/ReGraM/model_checkpoints')

OUTPUT_PATH = (
    EXP_ROOT
    / 'reports'
    / 'relation_probe'
    / f'{CATEGORY}_position_shift_{SEVERITY}_{COMPONENT_MODEL}_known_transform.json'
)

def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')

def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [
        parent / 'content' / dataset_name,
        parent / 'data' / 'row' / dataset_name,
    ]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct

def load_position_entries() -> list[dict]:
    entries = []
    for line in POSITION_MANIFEST.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        entry = json.loads(line)
        if (
            entry.get('category') == CATEGORY
            and entry.get('augmentation_type') == 'position_shift'
            and entry.get('severity') == SEVERITY
        ):
            entries.append(entry)
        if len(entries) >= LIMIT:
            break
    return entries

def resolve_source_path(entry: dict) -> Path:
    path = Path(entry['source_path'])
    if path.is_absolute():
        return path
    if entry.get('source_path_mode') == 'repo_relative':
        return REPO_ROOT / path
    return path.resolve()

def expected_grounding_mask_path_for_source(image_path: Path, *, mask_root: Path) -> Path:
    try:
        rel_path = image_path.relative_to(GROUNDING_MASK_DATA_ROOT / CATEGORY)
    except ValueError:
        parts = list(image_path.parts)
        if CATEGORY not in parts:
            raise
        category_index = len(parts) - 1 - parts[::-1].index(CATEGORY)
        rel_path = Path(*parts[category_index + 1 :])
    return mask_root / CATEGORY / rel_path.with_suffix('') / 'grounding_mask.png'

def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT, RAW_CATEGORY_GOOD, GROUNDING_MASK_DATA_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    RAW_CATEGORY_GOOD = RAW_LOCO_ROOT / CATEGORY / 'test' / 'good'
    GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT
    if RAW_CATEGORY_GOOD.exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    RAW_CATEGORY_GOOD = RAW_LOCO_ROOT / CATEGORY / 'test' / 'good'
    GROUNDING_MASK_DATA_ROOT = RAW_LOCO_ROOT

def copy_required_grounding_masks_from_drive_if_needed(entries: list[dict]) -> list[Path]:
    if COMPONENT_MODEL != 'grounding_mask_cluster':
        return []
    session_paths = [
        expected_grounding_mask_path_for_source(resolve_source_path(entry), mask_root=GROUNDING_MASK_ROOT)
        for entry in entries
    ]
    missing_session_paths = [path for path in session_paths if not path.exists()]
    if not missing_session_paths:
        print(f'grounding masks already present in session: {len(session_paths)} files')
        return session_paths

    mount_drive_if_available()
    copied = 0
    still_missing = []
    for index, entry in enumerate(entries, start=1):
        source_path = resolve_source_path(entry)
        session_mask_path = expected_grounding_mask_path_for_source(source_path, mask_root=GROUNDING_MASK_ROOT)
        if session_mask_path.exists():
            continue
        drive_mask_path = expected_grounding_mask_path_for_source(source_path, mask_root=DRIVE_MASK_ROOT)
        if not drive_mask_path.exists():
            still_missing.append(drive_mask_path)
            continue
        session_mask_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_mask_path, session_mask_path)
        copied += 1
        if copied == 1 or copied % 10 == 0 or index == len(entries):
            print(f'copied grounding masks from Drive: {copied}/{len(missing_session_paths)}')

    if still_missing:
        preview = still_missing[:5]
        suffix = f' ... +{len(still_missing) - 5} more' if len(still_missing) > 5 else ''
        raise FileNotFoundError(f'Missing required grounding masks in Drive: {preview}{suffix}')
    return session_paths

def restore_sam_checkpoint_from_drive_if_needed() -> None:
    if COMPONENT_MODEL != 'sam_lad' or SAM_CHECKPOINT.exists():
        return
    mount_drive_if_available()
    cached_checkpoint = DRIVE_MODEL_CHECKPOINT_ROOT / 'sam_hq_vit_h.pth'
    if cached_checkpoint.exists():
        SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['cp', '-n', str(cached_checkpoint), str(SAM_CHECKPOINT)], check=True)

restore_raw_loco_from_drive_if_needed()
POSITION_ENTRIES = load_position_entries()
REQUIRED_GROUNDING_MASKS = copy_required_grounding_masks_from_drive_if_needed(POSITION_ENTRIES)
restore_sam_checkpoint_from_drive_if_needed()
GROUNDING_MASK_CATEGORY_ROOT = GROUNDING_MASK_ROOT / CATEGORY / 'test' / 'good'

required_paths = {
    'relation runner': SRC_ROOT / 'relation' / 'run_relation_probe.py',
    'position manifest': POSITION_MANIFEST,
    'raw LOCO test/good images': RAW_CATEGORY_GOOD,
}
if COMPONENT_MODEL == 'grounding_mask_cluster':
    required_paths.update({
        'grounding mask category root': GROUNDING_MASK_CATEGORY_ROOT,
        'grounding mask cluster config': GROUNDING_CLUSTER_CONFIG,
    })
elif COMPONENT_MODEL == 'sam_lad':
    required_paths.update({
        'SAM checkpoint': SAM_CHECKPOINT,
        'segment_anything package': SAM_ROOT / 'segment_anything',
    })
else:
    raise ValueError(f'Unsupported COMPONENT_MODEL={COMPONENT_MODEL}')

missing_paths = {name: path for name, path in required_paths.items() if not path.exists()}
missing_masks = [path for path in REQUIRED_GROUNDING_MASKS if not path.exists()]
if missing_paths or missing_masks:
    raise FileNotFoundError(
        f'Missing required {COMPONENT_MODEL} relation-probe path(s): {missing_paths}; '
        f'missing_masks={missing_masks[:5]}'
        + (f' ... +{len(missing_masks) - 5} more' if len(missing_masks) > 5 else '')
        + '. Run 01 setup if Drive caches are not available.'
    )

settings = {
    'category': CATEGORY,
    'severity': SEVERITY,
    'component_model': COMPONENT_MODEL,
    'limit': LIMIT,
    'selected_entries': len(POSITION_ENTRIES),
    'max_components': MAX_COMPONENTS,
    'progress_every': PROGRESS_EVERY,
    'raw_loco_root': str(RAW_LOCO_ROOT),
    'output': str(OUTPUT_PATH),
}
if COMPONENT_MODEL == 'grounding_mask_cluster':
    settings.update({
        'mask_root': str(GROUNDING_MASK_ROOT),
        'mask_data_root': str(GROUNDING_MASK_DATA_ROOT),
        'cluster_config': str(GROUNDING_CLUSTER_CONFIG),
        'selected_mask_count': len(REQUIRED_GROUNDING_MASKS),
        'mask_category_root': str(GROUNDING_MASK_CATEGORY_ROOT),
    })
else:
    settings.update({
        'sam_points_per_side': SAM_POINTS_PER_SIDE,
        'sam_crop_n_layers': SAM_CROP_N_LAYERS,
        'sam_checkpoint': str(SAM_CHECKPOINT),
        'sam_root': str(SAM_ROOT),
    })

display(pd.DataFrame([settings]))


## Cell Role: Run Local Tests

선택한 component preprocessing과 relation geometry unit test를 먼저 확인한다. `grounding_mask_cluster`는 colored mask를 runner에 연결하는 integration test까지 포함한다.


In [ ]:
test_targets = [
    'experiments.validation.condition_shift_baseline.tests.test_relation_geometry',
]
if COMPONENT_MODEL == 'grounding_mask_cluster':
    test_targets.append('experiments.validation.condition_shift_baseline.tests.test_grounding_mask_cluster')
else:
    test_targets.append('experiments.validation.condition_shift_baseline.tests.test_sam_lad_components')

subprocess.run(
    [sys.executable, '-m', 'unittest', *test_targets],
    cwd=REPO_ROOT,
    check=True,
)


## Cell Role: Run Relation Probe

선택한 component model로 component node를 만들고, known position transform relation score를 계산한다. `grounding_mask_cluster`는 기존 `grounding_mask.png`를 색 라벨별 raw mask로 분해한 뒤 thing/stuff/small-isolated node로 정규화한다.


In [ ]:
cmd = [
    sys.executable,
    str(SRC_ROOT / 'relation' / 'run_relation_probe.py'),
    '--repo-root', str(REPO_ROOT),
    '--manifest', str(POSITION_MANIFEST),
    '--category', CATEGORY,
    '--severity', SEVERITY,
    '--limit', str(LIMIT),
    '--max-components', str(MAX_COMPONENTS),
    '--component-model', COMPONENT_MODEL,
    '--progress-every', str(PROGRESS_EVERY),
    '--output', str(OUTPUT_PATH),
]

if COMPONENT_MODEL == 'grounding_mask_cluster':
    cmd.extend([
        '--mask-root', str(GROUNDING_MASK_ROOT),
        '--mask-data-root', str(GROUNDING_MASK_DATA_ROOT),
        '--grounding-cluster-config', str(GROUNDING_CLUSTER_CONFIG),
    ])
elif COMPONENT_MODEL == 'sam_lad':
    cmd.extend([
        '--sam-checkpoint', str(SAM_CHECKPOINT),
        '--sam-root', str(SAM_ROOT),
        '--sam-device', SAM_DEVICE,
        '--sam-min-area-ratio', str(SAM_MIN_AREA_RATIO),
        '--sam-small-area-ratio', str(SAM_SMALL_AREA_RATIO),
        '--sam-max-area-ratio', str(SAM_MAX_AREA_RATIO),
        '--sam-points-per-side', str(SAM_POINTS_PER_SIDE),
        '--sam-crop-n-layers', str(SAM_CROP_N_LAYERS),
    ])

print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Cell Role: Inspect Scores And Nodes

component 수, mask/node type 분포, relation score median을 확인한다. `grounding_mask_cluster` 결과는 `component_nodes`에 thing/stuff/small-isolated 내부 통계를 보존한다.


In [ ]:
summary = json.loads(OUTPUT_PATH.read_text())
metric_df = (
    pd.DataFrame([summary['metric_medians']])
    .T
    .reset_index()
    .rename(columns={'index': 'metric', 0: 'median'})
)
display(metric_df)

rows_df = pd.DataFrame(summary['rows'])
preview_cols = [
    'source_id',
    'component_model',
    'component_source',
    'component_count',
    'raw_mask_count',
    'thing_count',
    'stuff_cluster_count',
    'small_isolated_count',
    'merged_small_count',
    'component_sources',
    'component_note',
    's_abs',
    's_centered_norm',
    's_pair_norm',
]
display(rows_df[[col for col in preview_cols if col in rows_df.columns]].head(10))

node_rows = []
for row in summary['rows'][:3]:
    for node in row.get('component_nodes', [])[:8]:
        node_rows.append({
            'source_id': row.get('source_id'),
            'node_id': node.get('node_id'),
            'node_type': node.get('node_type'),
            'num_members': node.get('num_members'),
            'area_ratio': node.get('area_ratio'),
            'mask_ids': node.get('mask_ids'),
            'density': node.get('member_stats', {}).get('density'),
            'spatial_dispersion_ratio': node.get('member_stats', {}).get('spatial_dispersion_ratio'),
            'reason': node.get('debug', {}).get('reason'),
        })
if node_rows:
    display(pd.DataFrame(node_rows))

if summary['skipped_count']:
    display(pd.DataFrame(summary['skipped']).head(10))


## Cell Role: Visualize Cluster Nodes

원본 이미지, grounding mask label을 밝게 재색칠한 view, grounding mask 위 clustered node overlay, 원본 위 clustered node overlay를 함께 확인한다. 실제 mask 값은 바꾸지 않고 visualization에서만 unique color label을 palette로 재매핑한다.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import numpy as np

VIS_ROW_INDEX = 0
VIS_MAX_NODES = 24
NODE_COLORS = {
    'thing': '#1f77b4',
    'stuff_cluster': '#ff7f0e',
    'small_isolated': '#d62728',
}
LABEL_PALETTE = np.asarray([
    [31, 119, 180],
    [255, 127, 14],
    [44, 160, 44],
    [214, 39, 40],
    [148, 103, 189],
    [140, 86, 75],
    [227, 119, 194],
    [127, 127, 127],
    [188, 189, 34],
    [23, 190, 207],
    [57, 59, 121],
    [82, 84, 163],
    [107, 110, 207],
    [156, 158, 222],
    [99, 121, 57],
    [140, 162, 82],
    [181, 207, 107],
    [206, 219, 156],
    [140, 109, 49],
    [189, 158, 57],
    [231, 186, 82],
    [231, 203, 148],
], dtype=np.uint8)

def recolor_label_mask(mask_image: np.ndarray) -> np.ndarray:
    flat = mask_image.reshape(-1, 3)
    colors, inverse = np.unique(flat, axis=0, return_inverse=True)
    recolored_colors = np.zeros_like(colors, dtype=np.uint8)
    palette_index = 0
    for color_index, color in enumerate(colors):
        if np.all(color == 0):
            recolored_colors[color_index] = [0, 0, 0]
            continue
        recolored_colors[color_index] = LABEL_PALETTE[palette_index % len(LABEL_PALETTE)]
        palette_index += 1
    return recolored_colors[inverse].reshape(mask_image.shape)

def draw_node_boxes(ax, nodes: list[dict], *, max_nodes: int) -> None:
    for node in nodes[:max_nodes]:
        x1, y1, x2, y2 = node['bbox']
        cx, cy = node['centroid']
        node_type = node['node_type']
        color = NODE_COLORS.get(node_type, 'white')
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
        )
        ax.scatter([cx], [cy], c=[color], s=28)
        ax.text(
            x1,
            max(0, y1 - 3),
            f"{node['node_id']} {node_type} n={node['num_members']}",
            color=color,
            fontsize=8,
            bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 1, 'edgecolor': 'none'},
        )

def draw_cluster_nodes(row_index: int = 0, max_nodes: int = 24) -> None:
    if not summary['rows']:
        raise ValueError('No evaluated rows to visualize')
    row = summary['rows'][row_index]
    image_path = Path(row['source_path'])
    image = np.asarray(Image.open(image_path).convert('RGB'))
    mask_path = expected_grounding_mask_path_for_source(image_path, mask_root=GROUNDING_MASK_ROOT)
    mask_image = None
    mask_view = None
    if mask_path.exists():
        mask_image = np.asarray(Image.open(mask_path).convert('RGB'))
        mask_view = recolor_label_mask(mask_image)

    nodes = row.get('component_nodes', [])
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    axes[0].imshow(image)
    axes[0].set_title(f"original: {row.get('source_id')}")
    axes[0].axis('off')

    if mask_view is not None:
        axes[1].imshow(mask_view)
        axes[1].set_title('grounding mask labels recolored')
    else:
        axes[1].imshow(image)
        axes[1].set_title(f'missing mask: {mask_path}')
    axes[1].axis('off')

    if mask_view is not None:
        axes[2].imshow(mask_view)
        axes[2].set_title('nodes on recolored mask')
    else:
        axes[2].imshow(image)
        axes[2].set_title('nodes on image; mask missing')
    draw_node_boxes(axes[2], nodes, max_nodes=max_nodes)
    axes[2].axis('off')

    axes[3].imshow(image)
    axes[3].set_title('nodes on original')
    draw_node_boxes(axes[3], nodes, max_nodes=max_nodes)
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

    if mask_image is not None:
        nonzero_color_count = int(len(np.unique(mask_image.reshape(-1, 3), axis=0)) - 1)
        print(f'mask_path={mask_path}')
        print(f'nonzero_label_colors={nonzero_color_count}')

    if nodes:
        display(pd.DataFrame([
            {
                'node_id': node['node_id'],
                'node_type': node['node_type'],
                'num_members': node['num_members'],
                'area_ratio': node['area_ratio'],
                'bbox': node['bbox'],
                'mask_ids': node['mask_ids'],
                'density': node.get('member_stats', {}).get('density'),
                'spatial_dispersion_ratio': node.get('member_stats', {}).get('spatial_dispersion_ratio'),
                'reason': node.get('debug', {}).get('reason'),
            }
            for node in nodes[:max_nodes]
        ]))

draw_cluster_nodes(VIS_ROW_INDEX, VIS_MAX_NODES)
